# 📄 Scientific Paper Ingestion Pipeline

This notebook demonstrates the PDF ingestion process for scientific papers, including:
- Text extraction from PDFs
- Metadata extraction (title, author, year)
- Section-based text segmentation
- Visualization of extracted data

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

from ingestion import PaperIngestion
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

print("✓ Imports successful")

## 1. Initialize the Ingestion System

In [ ]:
# Initialize the paper ingestion system
ingestion = PaperIngestion()

print("Paper Ingestion System Initialized")
print(f"Supported section patterns: {len(ingestion.SECTION_PATTERNS)}")

## 2. Process a Single Paper

Let's process a sample PDF and extract its structure.

In [ ]:
# Path to your sample PDF
pdf_path = "../data/sample_paper.pdf"

# Check if file exists
if Path(pdf_path).exists():
    print(f"✓ Found: {pdf_path}")
    
    # Process the paper
    paper = ingestion.process_paper(pdf_path, paper_id="sample_001")
    
    print("\n📊 Extraction Results:")
    print(f"Paper ID: {paper['paper_id']}")
    print(f"Title: {paper['metadata']['title']}")
    print(f"Author: {paper['metadata']['author']}")
    print(f"Year: {paper['metadata']['year']}")
    print(f"Sections found: {len(paper['sections'])}")
else:
    print(f"⚠️ PDF not found at {pdf_path}")
    print("Please add a sample PDF to the data/ directory")
    
    # Create dummy data for demonstration
    paper = {
        'paper_id': 'demo_001',
        'metadata': {
            'title': 'Demo Paper: Machine Learning Applications',
            'author': 'John Doe et al.',
            'year': '2024',
            'filename': 'demo_paper.pdf'
        },
        'sections': {
            'abstract': 'This is a demonstration of the abstract section...',
            'introduction': 'In this paper, we introduce a novel approach...',
            'methods': 'Our methodology consists of three main steps...',
            'results': 'The experimental results demonstrate...',
            'conclusion': 'In conclusion, we have shown that...'
        },
        'full_text': 'Full text would be here...'
    }

## 3. Visualize Extracted Sections

Let's analyze the structure of the extracted paper.

In [ ]:
# Create a visualization of section lengths
sections_data = []
for section_name, section_text in paper['sections'].items():
    sections_data.append({
        'Section': section_name.capitalize(),
        'Characters': len(section_text),
        'Words': len(section_text.split())
    })

df_sections = pd.DataFrame(sections_data)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart - Characters
df_sections.plot(x='Section', y='Characters', kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Section Length (Characters)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Section')
axes[0].set_ylabel('Number of Characters')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend().remove()

# Bar chart - Words
df_sections.plot(x='Section', y='Words', kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Section Length (Words)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Section')
axes[1].set_ylabel('Number of Words')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend().remove()

plt.tight_layout()
plt.show()

print("\n📊 Section Statistics:")
print(df_sections.to_string(index=False))

## 4. Display Section Content

Preview the content of each section.

In [ ]:
# Display first 200 characters of each section
print("📝 Section Previews:\n")
for section_name, section_text in paper['sections'].items():
    print(f"{'='*60}")
    print(f"📌 {section_name.upper()}")
    print(f"{'='*60}")
    preview = section_text[:200].strip()
    print(f"{preview}...")
    print()

## 5. Process Multiple Papers

Batch processing example.

In [ ]:
# Find all PDFs in data directory
data_dir = Path("../data")
pdf_files = list(data_dir.glob("*.pdf"))

if pdf_files:
    print(f"Found {len(pdf_files)} PDF file(s)")
    
    # Process all papers
    papers = ingestion.process_multiple_papers([str(p) for p in pdf_files])
    
    # Save to JSON
    output_path = "../data/processed_papers.json"
    ingestion.save_to_json(papers, output_path)
    
    print(f"\n✓ Processed {len(papers)} papers")
    print(f"✓ Saved to {output_path}")
else:
    print("No PDF files found in ../data/")
    print("Creating example processed data...")
    
    # Create example data
    papers = [paper]  # Use the demo paper
    output_path = "../data/processed_papers.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(papers, f, indent=2, ensure_ascii=False)
    print(f"✓ Created example data at {output_path}")

## 6. Summary Statistics

Overview of all processed papers.

In [ ]:
# Create summary
summary_data = []
for p in papers:
    summary_data.append({
        'Paper ID': p['paper_id'],
        'Title': p['metadata']['title'][:50] + '...',
        'Year': p['metadata']['year'],
        'Sections': len(p['sections']),
        'Total Length': len(p['full_text'])
    })

df_summary = pd.DataFrame(summary_data)
print("📊 Papers Summary:\n")
print(df_summary.to_string(index=False))

print(f"\n✅ Total papers processed: {len(papers)}")
print(f"✅ Total sections extracted: {sum(len(p['sections']) for p in papers)}")

## Next Steps

✅ Papers are now extracted and structured  
⏭️ Next: Move to `02_chunking_embeddings.ipynb` to create text chunks and embeddings